# Fetching data from an API

We ask a web API for TV-show data, turn each reply into a DataFrame, loop over the pages, and save the result as a CSV file.

The API used in the Note's original example (TMDB) needs a personal key, so here we use **TVmaze** (`api.tvmaze.com`), which is free and needs no key. If there is no internet, every step falls back to saved copies in `data/`.

In [ ]:
# pandas builds tables; requests sends web requests
import json
import time

import pandas as pd
import requests

## 1. An API that needs a key

Without a key, TMDB refuses: status code 401 means "not authorised".

In [ ]:
tmdb_url = "https://api.themoviedb.org/3/movie/top_rated?language=en-US&page=1"
try:
    # send the request; give up after 10 seconds
    reply = requests.get(tmdb_url, timeout=10)
    # the status code says how the request went
    print(reply.status_code)
    # the body explains why
    print(reply.json())
except requests.RequestException as error:
    print("No internet:", error)

## 2. One page from a keyless API

In [ ]:
url = "https://api.tvmaze.com/shows?page=0"
try:
    response = requests.get(url, timeout=10)
    # 200 means OK
    print(response.status_code)
    # turn the JSON text into Python lists and dictionaries
    shows = response.json()
except requests.RequestException:
    # no internet: use the reply we saved earlier
    with open("data/tvmaze_shows_page0.json") as f:
        shows = json.load(f)

# the reply is a list with one dictionary per show
print(type(shows), len(shows))

In [ ]:
# the first show: a dictionary, some values are dictionaries again
first = shows[0]
{key: first[key] for key in ["id", "name", "language", "premiered", "rating", "weight"]}

### A list of dictionaries becomes a DataFrame

In [ ]:
# one row per dictionary, one column per key
page_df = pd.DataFrame(shows)
print(page_df.shape)
# 'rating' still holds a small dictionary in each cell
page_df[["id", "name", "rating"]].head(3)

In [ ]:
# json_normalize opens nested dictionaries into their own columns,
# so {'average': 6.6} becomes a column 'rating.average'
page_df = pd.json_normalize(shows)

# keep only the columns we want
columns = ["id", "name", "language", "premiered",
           "rating.average", "weight", "summary"]
page_df = page_df[columns]
print(page_df.shape)
page_df.head()

## 3. Looping over pages

Each page gives one small DataFrame. We collect them in a list and join them once at the end.

In [ ]:
frames = []                      # one small DataFrame per page
for page in range(0, 5):         # range(0, 400) gets every page
    url = f"https://api.tvmaze.com/shows?page={page}"
    try:
        response = requests.get(url, timeout=10)
    except requests.RequestException:
        break                    # no internet: stop
    # 404 means there is no such page: we have read them all
    if response.status_code != 200:
        break
    temp_df = pd.json_normalize(response.json())[columns]
    frames.append(temp_df)
    # wait half a second, to stay under the API's rate limit
    time.sleep(0.5)

print(len(frames), "pages fetched")

In [ ]:
if frames:
    # ignore_index=True numbers the rows 0, 1, 2, ... again
    df = pd.concat(frames, ignore_index=True)
else:
    # no internet: use the table we saved earlier
    df = pd.read_csv("data/shows.csv")
print(df.shape)
df.tail()

### Why `ignore_index=True`

Without it, every page keeps its own row numbers 0, 1, 2, ..., so the same label appears once per page.

In [ ]:
if len(frames) >= 2:
    joined = pd.concat(frames[:2])
    # label 0 appears twice: once from each page
    print(joined.loc[0, "name"])

### The old way: `DataFrame.append`

Older code grew the table with `df = df.append(temp_df, ignore_index=True)`. pandas 2.0 removed `append`; it now fails:

In [ ]:
try:
    pd.DataFrame().append(page_df)
except AttributeError as error:
    print("AttributeError:", error)

## 4. A page that does not exist

In [ ]:
try:
    missing = requests.get("https://api.tvmaze.com/shows?page=999", timeout=10)
    # 404: not found
    print(missing.status_code)
except requests.RequestException as error:
    print("No internet:", error)

## 5. Saving the dataset

In [ ]:
# index=False: do not write the row numbers as an extra column
df.to_csv("data/shows.csv", index=False)
pd.read_csv("data/shows.csv").shape